In [1]:
# hack to change jupyter directory in notebooks for imports and handle asyncio
import os
from pathlib import Path
import pandas as pd
from ib_insync import util
if Path.cwd().parts[-1] == 'notebooks':
    root = Path.cwd().parent
    import nest_asyncio
    nest_asyncio.apply()
    util.startLoop()
    pd.options.display.max_columns = None
    pd.options.display.float_format = '{:,.2f}'.format # set float precision with comma
else:
    root = Path.cwd()
os.chdir(root)

# Logger
import logging
log = logging.getLogger('ib_log')

In [2]:
import pandas as pd

fp = root / 'data' / 'master' / 'zArchives'

df_opt_margin = pd.read_pickle(fp / 'nse_opt_margins.pkl')

df = df_opt_margin

# replace commission with INR 20 for large commission values
df.loc[df.commission > 20, 'commission'] = 20

cols = ['conId', 'symbol', 'ib_sym', 'expiry', 'strike', 'right', 'undPrice', 
        'dte', 'opt_iv', 'lastPrice', 'bid', 'ask', 'lot', 'margin', 'commission']
df = df[cols].assign(rom=(df.lastPrice*df.lot-df.commission)/df.margin/df.dte*365).sort_values('rom', ascending=False)

In [4]:
# Get relevant calls and puts
m = ((df.strike > df.undPrice) & (df.right == 'C')) | ((df.strike < df.undPrice) & (df.right == 'P'))
df = df[m]

In [5]:
# Get history
df_hist = pd.read_pickle(fp / 'nse_hists.pkl')

In [8]:
g = df_hist.groupby('symbol')

In [ ]:
import math
g.close.pct_change().expanding(1).std(ddof=0)*math.sqrt(252)

In [ ]:
import numpy as np
g.close.pct_change().apply(lambda x: np.log(1+x)).std()